# Khảo sát chi tiết RAG Core (Phase 3)

Notebook này phân tách các bước của hệ thống Agentic RAG pháp luật giao thông thành các bước nhỏ để dễ phân tích và hiểu rõ cơ chế hoạt động của Hybrid Search và LLM Generator.

## 1. Nạp các thư viện và cấu hình cần thiết

Chạy cell này để khai báo module RAG Core.

In [14]:
import sys
import os
from pathlib import Path

# Đảm bảo import được thư mục source
BASE_DIR = Path().resolve().parent if Path().resolve().name in ["notebooks", "tests"] else Path().resolve()
sys.path.insert(0, str(BASE_DIR / "source"))

from rag_core import TrafficHybridRetriever, LegalAnswerGenerator

print("[OK] Thư viện RAG đã sẵn sàng!")

[OK] Thư viện RAG đã sẵn sàng!


## 2. Thử nghiệm Truy xuất thông tin (Retrieval)

Ở bước này, chúng ta sẽ xem **TrafficHybridRetriever** hoạt động ra sao. Retriever sử dụng **Hybrid Search** (kết hợp Vector của Qdrant và Keyword của BM25) bằng thuật toán phân giải RRF (Reciprocal Rank Fusion).

*Lưu ý: Qdrant Docker Container cần đang chạy trên cổng 6334.*

In [15]:
# Khởi tạo Retriever
retriever = TrafficHybridRetriever()

# Thử nghiệm 1 câu hỏi đang bị nhiễu (khiến Vector dễ nhầm)
query = "Chu kỳ đăng kiểm lần đầu cho xe ô tô con không kinh doanh vận tải sản xuất năm 2025?"

print(f"\nCÂU HỎI:\n{query}")
print("-" * 80)

# Lấy 5 chunks liên quan nhất
chunks = retriever.get_relevant_chunks(query, top_k=5)

print(f"TÌM THẤY {len(chunks)} ĐOẠN VĂN MẪU:\n")
for i, chunk in enumerate(chunks, 1):
    doc_id = chunk.metadata.get('doc_id', '?')
    dieu = chunk.metadata.get('dieu', '?')
    print(f"[{i}] Điểm RRF: {chunk.score:.4f} | Văn bản: {doc_id} | Điều {dieu}")
    print(f"    Preview: {chunk.content[:150]}...\n")

/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/source/rag_core/retriever.py:80: UserWarning: Qdrant client version 1.14.2 is incompatible with server version 1.17.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  self.client = QdrantClient(host=qdrant_host, port=qdrant_port)



CÂU HỎI:
Chu kỳ đăng kiểm lần đầu cho xe ô tô con không kinh doanh vận tải sản xuất năm 2025?
--------------------------------------------------------------------------------
TÌM THẤY 5 ĐOẠN VĂN MẪU:

[1] Điểm RRF: 0.0271 | Văn bản: 10/2020/NĐ-CP | Điều 13
    Preview: Văn bản: Nghị định 10/2020/NĐ-CP (Kinh doanh vận tải bằng xe ô tô) | Điều 13: Điều kiện kinh doanh vận tải hành khách bằng xe ô tô
b) Xe ô tô kinh doa...

[2] Điểm RRF: 0.0229 | Văn bản: 79/2024/TT-BCA | Điều 36
    Preview: Văn bản: Thông tư 79/2024/TT-BCA (Đăng ký xe) | Điều 36: Xác định năm sản xuất của xe
## Điều 36. Xác định năm sản xuất của xe
1. Năm sản xuất của xe ...

[3] Điểm RRF: 0.0164 | Văn bản: 10/2020/NĐ-CP | Điều 3
    Preview: Văn bản: Nghị định 10/2020/NĐ-CP (Kinh doanh vận tải bằng xe ô tô) | Điều 3: Giải thích từ ngữ
10. Trọng tải thiết kế của xe ô tô là số người và khối ...

[4] Điểm RRF: 0.0164 | Văn bản: 47/2024/TT-BGTVT | Điều 35
    Preview: Văn bản: Thông tư 47/2024/TT-BGTVT (Trạm kiểm định khí

## 3. Khởi tạo LLM Generator (LangChain)

Tiếp theo, chúng ta cắm một LLM vào thay cho con người để hệ thống tự đọc văn bản được truy xuất ở trên và sinh ra câu trả lời.

Bạn cần thiết lập khóa API cho OpenAI hoặc Google :

In [ ]:
import getpass

provider = "google"  # Chuyển đổi giữa "openai" hoặc "google"

if provider == "openai":
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Nhập OPENAI_API_KEY: ")
    generator = LegalAnswerGenerator(provider="openai", model="gpt-4o-mini")
    print("Đã khởi tạo OpenAI (GPT-4o-Mini)!")
else:
    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Nhập GOOGLE_API_KEY: ")
    # Lưu ý: "gemini-3.1-flash-lite" không tồn tại (API trả 404).
    # Dòng lite thật là "gemini-2.5-flash-lite" — rẻ/nhanh hơn gemini-2.5-flash.
    generator = LegalAnswerGenerator(provider="google", model="gemini-2.5-flash-lite")
    print("Đã khởi tạo Google Gemini (2.5 Flash Lite)!")

## 4. Xây dựng câu trả lời (LLM Generation)

Chúng ta truyền trực tiếp kết quả `chunks` lấy được từ `retriever` vào `generator`. Model sẽ đọc các chunk đó, định dạng lại thành số liệu và trích dẫn chuẩn pháp luật Việt Nam.

In [17]:
print(f"Gửi ngữ cảnh đến {generator.model_name}...")

# Chuyển đổi đối tượng data format
chunk_dicts = [c.to_dict() for c in chunks]

# Yêu cầu LLM trả lời
out = generator.generate(query, chunk_dicts)

print("\n========================== ĐÁP ÁN TỪ AI ==========================\n")
print(out['answer'])
print("\n========================== SIÊU DỮ LIỆU CẮM UI ==================\n")
print("Liệt kê các dòng trích dẫn gốc:\n")
for s in out['sources']:
    print(f" - Điều {s.get('dieu', '?')} - {s.get('ten_van_ban', '')} ({s.get('doc_id')})")

if out['refused']:
    print("\n[Cảnh báo]: LLM đã từ chối trả lời do thiếu thông tin pháp lý trong Context!")

Gửi ngữ cảnh đến gemini-3.1-flash-lite...


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised NotFound: 404 models/gemini-3.1-flash-lite is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods..


NotFound: 404 models/gemini-3.1-flash-lite is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.

## 5. Batch Evaluation (Test Suite)

Mục này chạy một loạt truy vấn khó trên pipeline đã khởi tạo ở các section trên (`retriever` + `generator`) để đánh giá độ chính xác trên nhiều nhóm tài liệu:

- **Xử phạt** (NĐ 168/2024): mức phạt vượt đèn đỏ, nồng độ cồn, số điểm GPLX bị trừ.
- **Kỹ thuật** (TT 47/2024 và các TT về cải tạo xe): chu kỳ đăng kiểm, điều kiện cải tạo.
- **Thủ tục hành chính** (TT 79/2024): đăng ký xe online.
- **Kịch bản đa văn bản**: LLM phải tổng hợp Luật 36/2024/QH15 và NĐ 168/2024 trong cùng một câu trả lời.
- **Ngoài phạm vi** (Bộ luật Hình sự): kỳ vọng `refused=True` và câu từ chối _"Thông tin này không có trong tài liệu được cung cấp."_

Điều kiện tiên quyết: các cell Section 1–4 phải đã chạy thành công để có sẵn biến `retriever` và `generator`.


In [ ]:
import time
import traceback

# Ghi chú retrieval: corpus KHÔNG chứa cụm khẩu ngữ "vượt đèn đỏ" (0 hit BM25);
# thuật ngữ pháp lý dùng trong NĐ 168/2024 là "không chấp hành hiệu lệnh của đèn
# tín hiệu giao thông". Ta để query ở dạng tự nhiên nhưng bổ sung keyword pháp lý
# để retriever có match; và bật top_k cao hơn để bù cho vocab mismatch.
TOP_K = 8

TEST_QUERIES = [
    {
        "category": "Xử phạt - Vượt đèn đỏ",
        "query": (
            "Vượt đèn đỏ (không chấp hành hiệu lệnh của đèn tín hiệu giao thông) "
            "khi điều khiển ô tô bị phạt bao nhiêu tiền và trừ bao nhiêu điểm GPLX "
            "theo Nghị định 168/2024/NĐ-CP?"
        ),
    },
    {
        "category": "Xử phạt - Nồng độ cồn",
        "query": "Mức phạt đối với người điều khiển xe máy có nồng độ cồn vượt quá 0,4 miligam/1 lít khí thở là bao nhiêu?",
    },
    {
        "category": "Kỹ thuật - Chu kỳ đăng kiểm",
        "query": "Chu kỳ kiểm định định kỳ của xe ô tô chở người đến 9 chỗ kinh doanh vận tải đã sản xuất trên 5 năm là bao lâu?",
    },
    {
        "category": "Kỹ thuật - Cải tạo xe",
        "query": "Điều kiện và hồ sơ để cải tạo xe cơ giới gồm những gì?",
    },
    {
        "category": "Thủ tục - Đăng ký xe online",
        "query": "Hồ sơ đăng ký xe lần đầu qua dịch vụ công trực tuyến cần những giấy tờ gì?",
    },
    {
        "category": "Tổng hợp đa văn bản",
        "query": "Khi Luật Trật tự, an toàn giao thông đường bộ 2024 có hiệu lực, hành vi không chấp hành hiệu lệnh đèn tín hiệu bị xử lý thế nào về hình thức phạt tiền và trừ điểm GPLX?",
    },
    {
        "category": "Ngoài phạm vi (kỳ vọng từ chối)",
        "query": "Tội giết người theo Bộ luật Hình sự bị xử phạt như thế nào?",
    },
]

# Gemini free tier giới hạn 5 request/phút → chờ ~15s giữa các query để tránh 429.
SLEEP_BETWEEN_QUERIES = 15

for i, case in enumerate(TEST_QUERIES, 1):
    print("=" * 90)
    print(f"[TEST {i}/{len(TEST_QUERIES)}] {case['category']}")
    print(f"CÂU HỎI: {case['query']}")
    print("-" * 90)

    try:
        chunks = retriever.get_relevant_chunks(case["query"], top_k=TOP_K)
        print(f"[retrieved {len(chunks)} chunks] top doc_ids: "
              f"{[c.metadata.get('doc_id') for c in chunks]}")

        result = generator.generate(case["query"], [c.to_dict() for c in chunks])

        print("\nTRẢ LỜI:")
        print(result["answer"])

        print("\nTRÍCH DẪN:")
        if result["sources"]:
            for s in result["sources"]:
                parts = [f"{s.get('doc_id', '?')}", f"Điều {s.get('dieu', '?')}"]
                if s.get("khoan") is not None:
                    parts.append(f"Khoản {s['khoan']}")
                if s.get("diem") is not None:
                    parts.append(f"Điểm {s['diem']}")
                ten = s.get("ten_van_ban", "")
                print(f"  - {' | '.join(parts)} — {ten}")
        else:
            print("  (không có trích dẫn)")

        print(f"REFUSED FLAG: {result['refused']}")
    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")
        traceback.print_exc(limit=2)

    print()
    if i < len(TEST_QUERIES):
        time.sleep(SLEEP_BETWEEN_QUERIES)
